## NISRINA ATIQAH DWIPUTRI RIDZKI (5027231075)

### Import Libraries

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from collections import OrderedDict

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
plt.style.use('seaborn-v0_8-whitegrid')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

class CFG:
    COMP_PATH = "/kaggle/input/llm-classification-finetuning"
    MODEL_PATH = "/kaggle/input/huggingfacedebertav3variants/deberta-v3-base" 
    MAX_LENGTH = 512
    BATCH_SIZE = 8
    EPOCHS = 1
    LEARNING_RATE = 1.5e-5
    SEED = 42
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_LABELS = 3

# Fungsi untuk seeding agar hasil reproducible
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)
print(f"Menggunakan device: {CFG.DEVICE}")

### Load Dataset

In [ ]:
train = pd.read_csv(os.path.join(CFG.COMP_PATH, "train.csv"))
test = pd.read_csv(os.path.join(CFG.COMP_PATH, "test.csv"))
submission = pd.read_csv(os.path.join(CFG.COMP_PATH, "sample_submission.csv"))

print("\nDataset berhasil dimuat!")

In [ ]:
print("Data Latih (train):")
print(f"Bentuk: {train.shape}")
train.head()

In [ ]:
print("\nData Uji (test):")
print(f"Bentuk: {test.shape}")
test.head()

In [ ]:
print("\nContoh File Submission:")
print(f"Bentuk: {submission.shape}")
submission.head()

In [ ]:
print("\nStatistik deskriptif data:")
train.describe().T

In [ ]:
print("\nNilai yang hilang :")
train.isnull().sum()

### EDA

In [ ]:
# Distribusi Target (Winner)
winner_counts = train[['winner_model_a', 'winner_model_b', 'winner_tie']].sum()

plt.figure(figsize=(10, 6))
sns.barplot(x=winner_counts.index, y=winner_counts.values, palette='viridis')
plt.title('Distribusi Pemenang (Model A vs Model B vs Tie)')
plt.ylabel('Jumlah')
plt.xlabel('Hasil')
plt.xticks([0, 1, 2], ['Model A Menang', 'Model B Menang', 'Seri'])
for i, count in enumerate(winner_counts.values):
    plt.text(i, count + 100, str(count), ha='center', fontweight='bold')
plt.show()

Visualisasi distribusi pemenang menunjukkan bahwa mayoritas pengguna lebih memilih Model A dibandingkan Model B, dengan jumlah yang cukup signifikan. Pilihan seri (tie) juga muncul namun dalam jumlah yang jauh lebih sedikit, mengindikasikan bahwa pengguna cenderung memiliki preferensi yang jelas antara dua respons yang diberikan. Ini mencerminkan adanya perbedaan kualitas yang dapat dibedakan secara konsisten antara kedua model, yang penting untuk dipertimbangkan dalam training dan evaluasi model.

In [ ]:
# Analisis Panjang Teks
train['prompt_len'] = train['prompt'].str.len()
train['response_a_len'] = train['response_a'].str.len()
train['response_b_len'] = train['response_b'].str.len()

plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.histplot(train['prompt_len'], bins=50, kde=True, color='blue')
plt.title('Distribusi Panjang Prompt')
plt.xlabel('Jumlah Karakter')

plt.subplot(1, 3, 2)
sns.histplot(train['response_a_len'], bins=50, kde=True, color='green')
plt.title('Distribusi Panjang Respons A')
plt.xlabel('Jumlah Karakter')

plt.subplot(1, 3, 3)
sns.histplot(train['response_b_len'], bins=50, kde=True, color='red')
plt.title('Distribusi Panjang Respons B')
plt.xlabel('Jumlah Karakter')

plt.tight_layout()
plt.show()

Analisis histogram terhadap panjang karakter dari prompt serta dua respons menunjukkan bahwa:

- Panjang prompt memiliki distribusi yang cukup beragam, namun cenderung berada di rentang menengah (tidak terlalu pendek, tidak terlalu panjang), menandakan bahwa pengguna memberikan instruksi yang cukup informatif.

- Respons A dan Respons B menunjukkan pola distribusi yang mirip, meskipun dalam beberapa kasus Response B terlihat sedikit lebih panjang. Hal ini bisa memberikan sinyal bahwa salah satu model cenderung menghasilkan jawaban yang lebih verbose, yang mungkin memengaruhi penilaian pengguna jika verbosity dianggap positif atau negatif tergantung konteks.

In [ ]:
# Frekuensi Model yang Diadu
model_counts = pd.concat([train['model_a'], train['model_b']]).value_counts().head(15)

plt.figure(figsize=(12, 8))
sns.barplot(y=model_counts.index, x=model_counts.values, palette='plasma', orient='h')
plt.title('Top 15 Model yang Paling Sering Diadu', fontsize=16)
plt.xlabel('Frekuensi', fontsize=12)
plt.ylabel('Nama Model', fontsize=12)
plt.show()

Analisis terhadap frekuensi model pada kolom model_a dan model_b menunjukkan bahwa distribusi pasangan model yang diuji tidak merata. Beberapa model seperti gpt-3.5-turbo dan gpt-4-turbo muncul secara signifikan lebih sering dibandingkan model lain. Misalnya, gpt-4-turbo sangat sering muncul sebagai model A dan dalam banyak kasus juga memenangkan perbandingan. Ini dapat menimbulkan bias evaluasi, terutama jika model tersebut lebih sering dibandingkan dengan model-model yang performanya jauh lebih lemah.

In [ ]:
# Word Cloud untuk Prompt
prompt_text = ' '.join(train['prompt'].dropna())
wordcloud = WordCloud(width=1000, height=500, background_color='black', colormap='cividis').generate(prompt_text)

plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud untuk Kata-kata Umum dalam Prompt', fontsize=16)
plt.show()
del prompt_text, wordcloud
gc.collect()

Word cloud yang dihasilkan memperlihatkan kata-kata paling sering muncul dalam prompt yang diberikan kepada model, di mana ukuran kata mencerminkan frekuensi relatifnya. Beberapa kata yang paling menonjol antara lain:

- "write", "answer", "one", "make", "want", "give", "use", "need"
Ini menunjukkan bahwa mayoritas prompt bersifat instruksional—pengguna cenderung meminta model untuk menulis sesuatu, memberikan jawaban, membuat sesuatu, atau melakukan tugas tertentu. Kata kerja imperatif seperti ini mengindikasikan bahwa prompt lebih banyak bersifat tugas eksplisit.

- "question", "name", "number", "time", "example", "information"
Banyak prompt tampaknya berisi permintaan untuk menjawab soal, memberi contoh, atau menyebutkan informasi spesifik. Hal ini konsisten dengan gaya prompt yang sering digunakan dalam pengujian kapabilitas reasoning dan factual recall dari LLM.

- "explain", "describe", "list", "based", "following"
Kata-kata ini mengarah ke prompt yang meminta penjelasan atau pembuatan daftar, menunjukkan fokus pada klarifikasi konsep atau struktur informasi.

- "character", "story", "code", "user", "data"
Ini mencerminkan keragaman jenis tugas—mulai dari penceritaan naratif, pemrograman, hingga tugas berbasis data. Artinya, prompt tidak hanya factual tapi juga kreatif dan teknikal.

In [ ]:
# Hubungan antara Panjang Respons dan Kemenangan (Verbosity Bias)
train['len_diff'] = train['response_a_len'] - train['response_b_len']
train['winner_label'] = np.select(
    [train['winner_model_a']==1, train['winner_model_b']==1, train['winner_tie']==1],
    ['model_a', 'model_b', 'tie']
)

plt.figure(figsize=(12, 7))
sns.boxplot(x='winner_label', y='len_diff', data=train, palette='coolwarm')
plt.title('Perbedaan Panjang Respons (A - B) vs Pemenang', fontsize=16)
plt.xlabel('Pemenang', fontsize=12)
plt.ylabel('Perbedaan Jumlah Kata (Respons A - Respons B)', fontsize=12)
plt.axhline(0, color='black', linestyle='--')
plt.show()

Ringkasan hasil EDA :
- Rata-rata `len_diff` mendekati **nol** untuk semua kategori pemenang (`model_a`, `model_b`, `tie`).
- **Tidak ada pola konsisten** bahwa respons yang lebih panjang lebih sering menang.
- **Outlier ekstrem** memang ada, tapi tidak memengaruhi distribusi utama.
- Kesimpulan: **Panjang respons bukan faktor dominan** dalam pemilihan pemenang.

### Preprocessing

In [ ]:
print("\nMemulai Preprocessing...")

# Fungsi untuk membuat input teks tunggal
def create_text_input(df):
    return "PROMPT: " + df["prompt"].fillna('') + \
           " [SEP] RESPONSE A: " + df["response_a"].fillna('') + \
           " [SEP] RESPONSE B: " + df["response_b"].fillna('')

train['text'] = create_text_input(train)
test['text'] = create_text_input(test)

train['target'] = np.select(
    [train['winner_model_a']==1, train['winner_model_b']==1, train['winner_tie']==1],
    [0, 1, 2]
)

print(f"Memuat tokenizer dari: {CFG.MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(
    CFG.MODEL_PATH,
    local_files_only=True 
)

# Bagi data menjadi training dan validasi
train_data, val_data = train_test_split(
    train,
    test_size=0.1,
    random_state=CFG.SEED,
    stratify=train['target']
)
print(f"Ukuran data training: {len(train_data)}")
print(f"Ukuran data validasi: {len(val_data)}")
print("Preprocessing selesai!")

### Modelling

In [ ]:
print("\nInisialisasi Model dan Dataset...")

# Membuat PyTorch Dataset Class 
class PreferenceDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.texts = df['text'].values
        self.is_test = is_test
        if not self.is_test:
            self.labels = df['target'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text,
            truncation=True,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        if not self.is_test:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

print(f"Memuat model dari path: {CFG.MODEL_PATH}")

config = AutoConfig.from_pretrained(CFG.MODEL_PATH)
config.num_labels = CFG.NUM_LABELS

model = AutoModelForSequenceClassification.from_pretrained(
    CFG.MODEL_PATH,
    config=config,
    ignore_mismatched_sizes=True,
    local_files_only=True
)

try:
    model.to(CFG.DEVICE)
    print(f"Model berhasil dimuat dan dipindahkan ke {CFG.DEVICE}")
except Exception as e:
    print(f"GAGAL MEMINDAHKAN MODEL KE GPU: {e}")

# Membuat dataset dan dataloader
train_dataset = PreferenceDataset(train_data, tokenizer, CFG.MAX_LENGTH)
val_dataset = PreferenceDataset(val_data, tokenizer, CFG.MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=2)

### Training

In [ ]:
print("\nMemulai Training...")

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LEARNING_RATE)
num_training_steps = len(train_loader) * CFG.EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0.1 * num_training_steps,
    num_training_steps=num_training_steps
)

# Loop Training
best_val_logloss = float('inf')
best_model_path = "best_model.pth"

for epoch in range(CFG.EPOCHS):
    print(f"======== Epoch {epoch + 1}/{CFG.EPOCHS} ========")
    model.train()
    total_train_loss = 0
    progress_bar = tqdm(train_loader, desc="Training")

    for batch in progress_bar:
        optimizer.zero_grad()
        inputs = {k: v.to(CFG.DEVICE) for k, v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
        progress_bar.set_postfix({'loss': loss.item()})

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Average Training Loss: {avg_train_loss:.4f}")

    # Validasi
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            labels = batch.pop('labels')
            inputs = {k: v.to(CFG.DEVICE) for k, v in batch.items()}
            outputs = model(**inputs)
            all_preds.append(outputs.logits.cpu().numpy())
            all_labels.append(labels.numpy())

    preds_np = np.concatenate(all_preds)
    labels_np = np.concatenate(all_labels)
    probs_np = torch.softmax(torch.tensor(preds_np), dim=1).numpy()

    val_logloss = log_loss(labels_np, probs_np)
    print(f"Validation LogLoss: {val_logloss:.4f}")

    if val_logloss < best_val_logloss:
        best_val_logloss = val_logloss
        torch.save(model.state_dict(), best_model_path)
        print(f"Model terbaik disimpan di '{best_model_path}' dengan LogLoss: {best_val_logloss:.4f}")

    gc.collect()

### Evaluate

In [ ]:
# Muat model terbaik untuk inferensi
model.load_state_dict(torch.load(best_model_path))
model.to(CFG.DEVICE)

print(f"\nSkor LogLoss terbaik pada set validasi adalah: {best_val_logloss:.4f}")

# Prediksi pada data test
test_dataset = PreferenceDataset(test, tokenizer, CFG.MAX_LENGTH, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=2)

def predict(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting on Test Data"):
            inputs = {k: v.to(CFG.DEVICE) for k, v in batch.items()}
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            all_preds.append(probs.cpu().numpy())
    return np.concatenate(all_preds)

predictions = predict(model, test_loader)

### Submission

In [ ]:
# Membuat file submission
submission['winner_model_a'] = predictions[:, 0]
submission['winner_model_b'] = predictions[:, 1]
submission['winner_tie'] = predictions[:, 2]

submission.to_csv("submission.csv", index=False)
print("\nFile submission.csv berhasil dibuat!")
submission.head()